# Direct Approach — Full-Text Embedding + HDBSCAN Clustering

## What this notebook does
Implements the **Direct Approach**: embedding full cybersecurity news articles using two
transformer-based models — **E5** (`intfloat/e5-base-v2`) and **SecureBERT**
(`ehsanaghaei/SecureBERT`) — then clustering the resulting vectors with HDBSCAN.

## What this notebook concludes
The Direct Approach **fails to produce coherent event-level clusters** regardless of model
choice or parameter tuning. E5 collapses 13,700+ articles into 2 clusters. SecureBERT
produces 3. Both yield near-zero silhouette scores. This is not a tuning failure —
it is an architectural one. Holistic article-level embeddings capture broad thematic
similarity, not event-level structure. This result directly motivates the Hybrid Approach
(see `02_hybrid_dependency_parsing.ipynb`).

## Pipeline
```
Raw Articles
    → Tokenization & Chunking (512 tokens, 96-token stride)
    → E5 / SecureBERT Embedding (chunk-level)
    → Token-Weighted Article-Level Aggregation
    → L2 Normalization
    → HDBSCAN Clustering
    → UMAP Visualization
```

## Models
| Model | Type | Pooling Strategy | Embedding Dim |
|-------|------|-----------------|---------------|
| `intfloat/e5-base-v2` | General-purpose, instruction-tuned | Masked mean | 768 |
| `ehsanaghaei/SecureBERT` | Cybersecurity domain-adapted | 4-layer masked mean | 768 |

## 1. Imports

In [ ]:
import hashlib
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import hdbscan
import umap
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score
from joblib import Parallel, delayed
from sklearn.model_selection import ParameterGrid

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Data Loading

The dataset consists of ~13,700 cybersecurity news articles provided by Sandia National
Laboratories, scraped from open-access sources via their internal tool, Ghost Trap.
See `data/README.md` for full dataset description.

Each article contains four attributes: `date`, `title`, `source`, and `article` (full text).

In [ ]:
# Load the cleaned dataset
# Preprocessing (null removal, deduplication, encoding cleanup) was
# performed in Alteryx Designer 2025.1.2.120 prior to this notebook
DATA_PATH = "../data/full_clean.xlsx"

df_raw = pd.read_excel(DATA_PATH)
df_raw["date"] = pd.to_datetime(df_raw["date"], errors="coerce", utc=True)

print(f"Total articles loaded: {len(df_raw):,}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head(3)

## 3. Article Formatting

Each row is converted to a dictionary with a stable `article_id` (MD5 hash of text),
enabling consistent cross-pipeline tracking without exposing raw indices.
Title is prepended to body text to anchor the semantic context of each article.

In [ ]:
def df_to_articles(df, include_title=True):
    """
    Convert dataframe rows to article dictionaries.

    Args:
        df: Raw dataframe with columns: title, article, source, date
        include_title: Prepend title to article body before embedding

    Returns:
        List of article dicts with keys: article_id, text, source, date, title
    """
    records = []
    for _, row in df.iterrows():
        text = f"{row['title']}\n\n{row['article']}" if include_title else row['article']
        # MD5 hash produces a stable, reproducible article ID across pipeline runs
        a_id = hashlib.md5(text.encode("utf-8", "ignore")).hexdigest()
        records.append({
            "article_id": a_id,
            "text": text,
            "source": row.get("source", ""),
            "date": row["date"],
            "title": row.get("title", ""),
        })
    return records


articles = df_to_articles(df_raw)
print(f"Articles formatted: {len(articles):,}")
print(f"Sample keys: {list(articles[0].keys())}")

## 4. Chunking

Transformer models are limited to 512 tokens. Articles frequently exceed this.
We use a sliding window of **512 tokens with a 96-token stride (~25% overlap)**
to preserve cross-sentence context at chunk boundaries.

**Key design decision:** Overlap is applied *within* articles only — chunks never
span article boundaries, preventing semantic contamination between documents.

This function is model-agnostic and reused for both E5 and SecureBERT.

In [ ]:
def chunk_article_by_tokens(text, tokenizer, max_len=512, stride=96):
    """
    Sliding window tokenization for a single article.

    Args:
        text: Raw article string
        tokenizer: HuggingFace tokenizer
        max_len: Maximum tokens per chunk (512 = BERT architecture limit)
        stride: Token overlap between adjacent chunks (~25% of max_len)

    Returns:
        List of chunk dicts with input_ids, attention_mask, and position metadata
    """
    if not text:
        return []

    enc = tokenizer(
        text,
        return_tensors=None,
        truncation=False,       # manual windowing only — do not auto-truncate
        padding=False,
        add_special_tokens=False
    )

    input_ids = enc["input_ids"]
    attn = [1] * len(input_ids)  # 1 = real token, 0 = padding

    windows = []
    start, chunk_id = 0, 0

    while start < len(input_ids):
        end = min(start + max_len, len(input_ids))
        windows.append({
            "chunk_id": chunk_id,
            "input_ids": input_ids[start:end],
            "attention_mask": attn[start:end],
            "start_token": start,
            "end_token": end
        })
        if end == len(input_ids):
            break
        start = end - stride  # slide forward, keeping stride-length overlap
        chunk_id += 1

    return windows


def chunk_corpus(articles, tokenizer, max_len=512, stride=96):
    """Apply chunking across all articles, attaching article metadata to each chunk."""
    all_chunks = []
    for art in articles:
        windows = chunk_article_by_tokens(art["text"], tokenizer, max_len, stride)
        for w in windows:
            w["article_id"] = art["article_id"]
            w["title"] = art.get("title", "")
            w["source"] = art.get("source", "")
            w["date"] = art.get("date", "")
            all_chunks.append(w)
    return all_chunks

## 5. Embedding Utilities

Two pooling strategies are used depending on the model:

- **Masked mean pooling** (E5): Average over real token positions, excluding padding.
  E5 is instruction-tuned for clustering tasks and does not require layer averaging.

- **4-layer masked mean pooling** (SecureBERT): Average the final 4 transformer
  hidden layers before pooling over token positions. SecureBERT lacks an explicit
  sentence embedding objective, so layer averaging stabilizes its representations
  for downstream clustering.

In [ ]:
def collate_windows_for_batch(tokenizer, windows, pad_to_multiple_of=8):
    """Pad a batch of chunk windows to uniform length for batch inference."""
    ids = [torch.tensor(w["input_ids"], dtype=torch.long) for w in windows]
    att = [torch.ones(len(w["input_ids"]), dtype=torch.long) for w in windows]
    return tokenizer.pad(
        {"input_ids": ids, "attention_mask": att},
        padding=True,
        return_tensors="pt",
        pad_to_multiple_of=pad_to_multiple_of
    )


@torch.no_grad()
def masked_mean_pool(last_hidden_state, attention_mask):
    """Mean pool over real (non-padded) token positions."""
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)  # [B, T, 1]
    summed = (last_hidden_state * mask).sum(dim=1)                  # [B, H]
    counts = mask.sum(dim=1).clamp(min=1e-9)                        # [B, 1]
    return summed / counts                                          # [B, H]


@torch.no_grad()
def last4_layers_masked_mean(out_hidden_states, attention_mask):
    """
    Average the final 4 transformer hidden layers before token-level pooling.
    Stabilizes representations for models without explicit sentence embedding training.
    """
    layers = out_hidden_states[-4:]                    # extract last 4 hidden layers
    stacked = torch.stack(layers, dim=0).mean(dim=0)  # average across layers → [B, T, H]
    return masked_mean_pool(stacked, attention_mask)


@torch.no_grad()
def embed_chunks(all_chunks, tokenizer, model, batch_size=64, pooling="masked_mean"):
    """
    Embed all chunks in batches and return a DataFrame with one row per chunk.

    Args:
        all_chunks: List of chunk dicts from chunk_corpus()
        tokenizer: HuggingFace tokenizer
        model: HuggingFace model
        batch_size: Chunks per forward pass
        pooling: 'masked_mean' (E5) or 'last4_masked_mean' (SecureBERT)

    Returns:
        DataFrame with one row per chunk and an 'embedding' column
    """
    use_last4 = (pooling == "last4_masked_mean")
    if use_last4:
        model.config.output_hidden_states = True

    rows, vecs = [], []

    for i in range(0, len(all_chunks), batch_size):
        batch_windows = all_chunks[i:i + batch_size]
        batch = collate_windows_for_batch(tokenizer, batch_windows)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        out = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=use_last4
        )

        if use_last4:
            pooled = last4_layers_masked_mean(out.hidden_states, attention_mask)
        else:
            pooled = masked_mean_pool(out.last_hidden_state, attention_mask)

        pooled = pooled.cpu().to(torch.float32).numpy()

        for j, w in enumerate(batch_windows):
            rows.append({
                "article_id": w["article_id"],
                "chunk_id": w["chunk_id"],
                "start_token": w["start_token"],
                "end_token": w["end_token"],
                "title": w.get("title", ""),
                "source": w.get("source", ""),
                "date": w.get("date", ""),
                "chunk_text": tokenizer.decode(w["input_ids"], skip_special_tokens=True)
            })
        vecs.append(pooled)

    X = np.vstack(vecs).astype("float32")
    df = pd.DataFrame(rows)
    df["embedding"] = [x.tolist() for x in X]
    return df


def aggregate_to_article_level(chunk_df):
    """
    Collapse chunk-level embeddings to one vector per article using
    token-weighted averaging.

    Chunks covering more tokens contribute proportionally more to the final
    representation, preventing short final chunks from being over-weighted
    in a simple mean.
    """
    def weighted_embed(group):
        vecs = np.stack(group["embedding"].to_numpy())                         # [n_chunks, dim]
        w = (group["end_token"] - group["start_token"]).to_numpy(dtype=float)
        w = np.clip(w, 1.0, None)    # guard against zero-length edge chunks
        w = w / w.sum()               # normalize weights to sum to 1
        return (vecs * w[:, None]).sum(axis=0)                                 # [dim]

    article_embeddings = (
        chunk_df.groupby("article_id")
        .apply(weighted_embed)
        .reset_index(name="embedding")
    )

    # Reattach metadata: use earliest date/title/source seen per article
    meta = (
        chunk_df.sort_values("date")
        .groupby("article_id")
        .agg(title=("title", "first"), source=("source", "first"), date=("date", "first"))
        .reset_index()
    )

    return article_embeddings.merge(meta, on="article_id", how="left")

## 6. E5 Embedding

`intfloat/e5-base-v2` is an instruction-tuned model optimized for clustering,
retrieval, and semantic similarity tasks. It produces compact, directionally
consistent 768-dimensional embeddings evaluated against the MTEB benchmark.

Unlike SecureBERT, E5 uses standard masked mean pooling — its training objective
already optimizes for embeddings suitable for clustering without layer averaging.

In [ ]:
E5_MODEL = "intfloat/e5-base-v2"

e5_tokenizer = AutoTokenizer.from_pretrained(E5_MODEL)
e5_model = AutoModel.from_pretrained(E5_MODEL)
e5_model.eval()
e5_model.to(device)

print(f"E5 loaded on {device}")

e5_chunks = chunk_corpus(articles, tokenizer=e5_tokenizer, max_len=512, stride=96)
print(f"Total E5 chunks: {len(e5_chunks):,}")
print(f"Average chunks per article: {len(e5_chunks)/len(articles):.1f}")

In [ ]:
# NOTE: Embedding is compute-heavy. Run on GPU or HPC (Alpine) for full corpus.
# Uncomment to re-generate from scratch:

# e5_chunk_df = embed_chunks(
#     e5_chunks, e5_tokenizer, e5_model,
#     batch_size=64, pooling="masked_mean"  # standard mean pool for E5
# )
# e5_chunk_df.to_parquet("../results/e5_chunks.parquet", index=False)
# print(f"Saved {len(e5_chunk_df):,} E5 chunk embeddings")

# Load pre-computed chunk embeddings (generated on CU Boulder Alpine HPC)
e5_chunk_df = pd.read_parquet("../results/e5_chunks.parquet")
e5_chunk_df["embedding"] = e5_chunk_df["embedding"].apply(np.array)

e5_article_df = aggregate_to_article_level(e5_chunk_df)
print(f"E5 article-level embeddings: {len(e5_article_df):,}")
print(f"Embedding dimension: {len(e5_article_df['embedding'].iloc[0])}")

## 7. SecureBERT Embedding

`ehsanaghaei/SecureBERT` is a domain-adapted BERT variant pre-trained on cybersecurity
corpora. It encodes technical terminology, CVE patterns, and threat language more
sensitively than general-purpose models.

**Pooling strategy:** 4-layer masked mean pooling over the final hidden states.
SecureBERT lacks an explicit sentence embedding training objective, so averaging
across the final 4 layers stabilizes its semantic signal before token-level pooling.

In [ ]:
SECUREBERT_MODEL = "ehsanaghaei/SecureBERT"

sb_tokenizer = AutoTokenizer.from_pretrained(SECUREBERT_MODEL)
sb_model = AutoModel.from_pretrained(SECUREBERT_MODEL, add_pooling_layer=False)
sb_model.eval()
sb_model.to(device)

print(f"SecureBERT loaded on {device}")

sb_chunks = chunk_corpus(articles, tokenizer=sb_tokenizer, max_len=512, stride=96)
print(f"Total SecureBERT chunks: {len(sb_chunks):,}")
print(f"Average chunks per article: {len(sb_chunks)/len(articles):.1f}")

In [ ]:
# Uncomment to re-generate from scratch:

# sb_chunk_df = embed_chunks(
#     sb_chunks, sb_tokenizer, sb_model,
#     batch_size=64, pooling="last4_masked_mean"  # 4-layer averaging for SecureBERT
# )
# sb_chunk_df.to_parquet("../results/securebert_chunks.parquet", index=False)
# print(f"Saved {len(sb_chunk_df):,} SecureBERT chunk embeddings")

# Load pre-computed chunk embeddings (generated on CU Boulder Alpine HPC)
sb_chunk_df = pd.read_parquet("../results/securebert_chunks.parquet")
sb_chunk_df["embedding"] = sb_chunk_df["embedding"].apply(np.array)

sb_article_df = aggregate_to_article_level(sb_chunk_df)
print(f"SecureBERT article-level embeddings: {len(sb_article_df):,}")
print(f"Embedding dimension: {len(sb_article_df['embedding'].iloc[0])}")

## 8. HDBSCAN Clustering

HDBSCAN was selected over K-means for three reasons:
1. Cybersecurity reporting is **non-uniform in density** — ransomware articles vastly
   outnumber niche CVE reports. K-means assumes equal-density spherical clusters.
2. HDBSCAN does **not require a predefined k** — cluster count emerges from the data.
3. HDBSCAN flags ambiguous articles as **noise (-1)** rather than forcing assignment.

All embeddings are L2-normalized before clustering so Euclidean distance reflects
directional similarity rather than magnitude.

In [ ]:
# Final parameters selected via grid search (Section 9)
MIN_CLUSTER_SIZE = 2
MIN_SAMPLES = 4
CLUSTER_SELECTION_EPS = 0.0001
# EPS: lower = more conservative merging, higher = more permissive cluster formation


def run_hdbscan(article_df, min_cluster_size, min_samples, eps):
    """
    Run HDBSCAN on article-level embeddings.

    Returns:
        result_df: article_df with cluster_id and membership_prob appended
        clusterer: fitted HDBSCAN object
        X_norm: L2-normalized embedding matrix
    """
    X = np.vstack(article_df["embedding"].values)
    X_norm = normalize(X, norm="l2")  # Euclidean on L2-normalized ≈ cosine distance

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        cluster_selection_epsilon=eps,
        metric="euclidean",
        gen_min_span_tree=True
    )
    clusterer.fit(X_norm)

    result_df = article_df.copy()
    result_df["cluster_id"] = clusterer.labels_
    result_df["membership_prob"] = clusterer.probabilities_

    return result_df, clusterer, X_norm


def print_clustering_summary(name, result_df, X_norm):
    """Print clustering diagnostics for a given model run."""
    labels = result_df["cluster_id"].values
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise_frac = (labels == -1).sum() / len(labels)
    sil = silhouette_score(X_norm, labels) if n_clusters > 1 else None

    print(f"\n{name} — Clustering Results")
    print(f"  Clusters found:   {n_clusters}")
    print(f"  Noise fraction:   {noise_frac:.4f}")
    if sil:
        print(f"  Silhouette score: {sil:.4f}")
    else:
        print(f"  Silhouette score: N/A")

    return n_clusters, noise_frac, sil

In [ ]:
# Run clustering — E5
e5_clustered, e5_clusterer, e5_X_norm = run_hdbscan(
    e5_article_df, MIN_CLUSTER_SIZE, MIN_SAMPLES, CLUSTER_SELECTION_EPS
)
e5_n, e5_noise, e5_sil = print_clustering_summary("E5 Direct", e5_clustered, e5_X_norm)

# Run clustering — SecureBERT
sb_clustered, sb_clusterer, sb_X_norm = run_hdbscan(
    sb_article_df, MIN_CLUSTER_SIZE, MIN_SAMPLES, CLUSTER_SELECTION_EPS
)
sb_n, sb_noise, sb_sil = print_clustering_summary("SecureBERT Direct", sb_clustered, sb_X_norm)

## 9. Parameter Tuning

Grid search over HDBSCAN parameters was run on the CU Boulder Alpine HPC cluster
using parallel workers (`joblib`). The same grid was applied to both models.

| Parameter | Values Tested | Rationale |
|-----------|---------------|-----------|
| `min_cluster_size` | 2, 4 | Lower bound = smallest meaningful event cluster |
| `min_samples` | 1, 2, 3, 4 | Range from permissive to ~log(n) conservative |
| `cluster_selection_epsilon` | 0.0001, 0.001, 0.01, 0.1 | Sensitivity to density merging threshold |

In [ ]:
def evaluate_hdbscan(X, min_cluster_size, min_samples, eps):
    """Evaluate a single HDBSCAN parameter combination. Designed for parallel execution."""
    try:
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            cluster_selection_epsilon=eps,
            metric="euclidean",
            gen_min_span_tree=True
        ).fit(X)

        labels = clusterer.labels_
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        noise_frac = np.sum(labels == -1) / len(labels)
        silhouette = silhouette_score(X, labels) if n_clusters > 1 else None

        return {
            "min_cluster_size": min_cluster_size,
            "min_samples": min_samples,
            "eps": eps,
            "n_clusters": n_clusters,
            "noise_frac": round(noise_frac, 3),
            "silhouette": round(silhouette, 4) if silhouette else None
        }
    except Exception as e:
        return {
            "min_cluster_size": min_cluster_size, "min_samples": min_samples,
            "eps": eps, "error": str(e)
        }


# Full grid search results were generated on Alpine HPC.
# To re-run, uncomment the relevant block:

# param_grid = {
#     "MIN_CLUSTER_SIZE": [2, 4],
#     "MIN_SAMPLES": [1, 2, 3, 4],
#     "CLUSTER_SELECTION_EPS": [0.0001, 0.001, 0.01, 0.1]
# }
# param_combinations = list(ParameterGrid(param_grid))

# E5 grid search:
# pd.DataFrame(Parallel(n_jobs=-1, verbose=10)(
#     delayed(evaluate_hdbscan)(e5_X_norm, p['MIN_CLUSTER_SIZE'], p['MIN_SAMPLES'], p['CLUSTER_SELECTION_EPS'])
#     for p in param_combinations
# )).to_csv("../results/e5_direct_tuning_results.csv", index=False)

# SecureBERT grid search:
# pd.DataFrame(Parallel(n_jobs=-1, verbose=10)(
#     delayed(evaluate_hdbscan)(sb_X_norm, p['MIN_CLUSTER_SIZE'], p['MIN_SAMPLES'], p['CLUSTER_SELECTION_EPS'])
#     for p in param_combinations
# )).to_csv("../results/securebert_direct_tuning_results.csv", index=False)

# Load saved results
e5_tuning_df = pd.read_csv("../results/e5_direct_tuning_results.csv")
sb_tuning_df = pd.read_csv("../results/securebert_direct_tuning_results.csv")

print("E5 — Top 5 parameter combinations by cluster count:")
display(e5_tuning_df.sort_values("n_clusters", ascending=False).head(5))

print("\nSecureBERT — Top 5 parameter combinations by cluster count:")
display(sb_tuning_df.sort_values("n_clusters", ascending=False).head(5))

## 10. UMAP Visualization

UMAP projects the 768-dimensional embedding space into 2D for visual inspection.
A well-functioning pipeline produces distinct, separated density pockets in UMAP space.

The Direct Approach produces a **homogeneous blob for both models** — articles occupy
a continuous, undifferentiated manifold with no separable event-level structure.
This is the clearest visual evidence that holistic embeddings are architecturally
insufficient for event-level clustering.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, X_norm, result_df, title in [
    (axes[0], e5_X_norm, e5_clustered, "E5 Direct — UMAP Projection"),
    (axes[1], sb_X_norm, sb_clustered, "SecureBERT Direct — UMAP Projection")
]:
    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
    proj = reducer.fit_transform(X_norm)
    labels = result_df["cluster_id"].values

    unique_labels = sorted(set(labels))
    colors = cm.tab20(np.linspace(0, 1, max(len(unique_labels), 1)))
    color_map = {label: colors[i] for i, label in enumerate(unique_labels)}
    color_map[-1] = [0.7, 0.7, 0.7, 0.3]  # noise = light gray, low opacity

    for label in unique_labels:
        mask = labels == label
        ax.scatter(proj[mask, 0], proj[mask, 1], c=[color_map[label]], s=2, alpha=0.5)

    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("UMAP Dimension 1")
    ax.set_ylabel("UMAP Dimension 2")
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle(
    "Direct Approach: Both models produce homogeneous manifolds — no event-level separation",
    fontsize=11, style="italic", y=1.01
)
plt.tight_layout()
plt.savefig("../results/direct_approach_umap_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: ../results/direct_approach_umap_comparison.png")

## 11. Results Summary

In [ ]:
summary = pd.DataFrame([
    {
        "Model": "Direct E5",
        "Clusters": e5_n,
        "Noise Fraction": round(e5_noise, 4),
        "Silhouette Score": round(e5_sil, 4) if e5_sil else "N/A",
        "Interpretation": "Corpus collapsed into 2 trivial clusters — no event signal"
    },
    {
        "Model": "Direct SecureBERT",
        "Clusters": sb_n,
        "Noise Fraction": round(sb_noise, 4),
        "Silhouette Score": round(sb_sil, 4) if sb_sil else "N/A",
        "Interpretation": "3 clusters — marginal improvement, no event-level coherence"
    }
])

print("Direct Approach — Final Results")
display(summary)

## 12. Conclusion

The Direct Approach fails regardless of model choice or parameter configuration.
SecureBERT's domain-specific pre-training on cybersecurity corpora does not
meaningfully improve over the general-purpose E5 — both collapse the corpus
into 2–3 trivial clusters with near-zero silhouette scores.

**The failure is architectural, not a tuning problem.**

Full-text embeddings capture broad thematic similarity. At the corpus level,
all cybersecurity articles look like each other. The model cannot distinguish
two articles about *different ransomware incidents* from two articles about
*the same incident reported by different outlets*. Event-level separation
requires relational structure: *who did what to whom*. Full-text embeddings
do not provide this — regardless of how domain-specific the model is.

**This result directly motivates the Hybrid Approach**, which injects event-level
structure via dependency parsing (Subject-Verb-Object extraction) before embedding,
bypassing the need for costly manual annotation.

→ Continue to `02_hybrid_dependency_parsing.ipynb`